# Stereo calibration — where the two cameras actually are

`rig.py` holds a `StereoRig`, `stereo.py` fuses two views into one pose, and
`validation/` renders ground truth for both. All of it has only ever run against a
**synthetic** rig built by `StereoRig.from_spherical(...)`. This notebook produces the real
one: ChArUco shots of the 9×6 board at many angles in, `stereo_rig.json` out.

**What it solves.** Per-camera intrinsics, then the camera-to-camera transform.

**World frame is camera A**, `T_world_camA = I`, with camera B carrying the relative pose.
This is the deliberate choice, not a default. Anchoring the world to the *board* would tie
the rig to a placement that only exists during calibration; anchoring it to the *robot* would
make the extrinsic depend on the very thing it is used to measure. Camera A is the only datum
that is both physically persistent and independent of the measurement. Rebasing onto the
robot's disk frame at rest is a separate, later step, taken at integration with the
visual-servoing pose estimator — and being a pure change of basis, it cannot disturb anything
solved here.

Consequence worth knowing: `rig.py`'s docstring promises world +z is up, and
`tilt_seen_deg(normal_world=(0,0,1))` assumes +z is the rotor axis. In the camera-A frame +z
is A's optical axis, so **`baseline_mm()` and `axis_separation_deg()` are valid** (both are
frame-independent) and the elevation-flavoured helpers are not, until that rebase happens.
The rig JSON records `world_frame: "camera_A"`.

**The board does not need to be fixed.** Each pair gives `T_camA_board` and `T_camB_board`
for the *same* physical board pose, so

$$T_{B \leftarrow A} \;=\; T_{B \leftarrow \text{board}} \; T_{\text{board} \leftarrow A}
\;=\; T_{B \leftarrow \text{board}} \, T_{A \leftarrow \text{board}}^{-1}$$

is independent of where the board was. Every pair is an independent estimate of the same
transform; the spread across pairs is the uncertainty, and an outlier is a bad *view*, not a
moved board. Sweep the board through many poses — that is what conditions the intrinsics.

**Units are millimetres everywhere**, matching `RADIUS_MM`, `baseline_mm` and every
`center_mm` column in the CSVs. The board is built in mm so `solvePnP` returns mm directly.
`vision/visual_servo.ipynb` builds it in metres — do not copy that here.

In [ ]:
import json
import math
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy.optimize import least_squares
from scipy.spatial.transform import Rotation

HERE = Path.cwd()
if not (HERE / "rig.py").exists():
    raise RuntimeError(f"open this notebook from controller/pose/, not {HERE}")
sys.path.insert(0, str(HERE))

import sources                             # noqa: E402
from rig import Camera, StereoRig          # noqa: E402
from zeroing import frame_from_normal      # noqa: E402  (reused by the self-test)

ESP32 = HERE.parents[1]                    # .../ESP32_PMW
VISION = HERE.parent / "vision"
OUT_DIR = ESP32 / "results" / "stereo_calibration"
PAIR_DIR = OUT_DIR / "pairs"
RIG_PATH = HERE / "stereo_rig.json"

print("cv2", cv2.__version__)
print("pairs ->", PAIR_DIR)
print("rig   ->", RIG_PATH)

## 1. The board

The 9×6 board is 150 × 100 mm on US Letter, `DICT_4X4_100`, 40 interior corners —
generated by `vision/visual_servo.ipynb` cell 2 and printed as
`vision/charuco_9x6_letter.pdf`.

**Scale is the error nothing downstream catches.** Tell the solver 16.667 mm when the
printer laid down 16.2 and every distance the rig reports is 2.9 % short, with a perfect
sub-pixel reprojection error to reassure you. This already happened once in this project:
`charuco_6x8.pdf` was exported with a 1080 × 1400 pt MediaBox (15 × 19.4 in), so every
printer rescaled it and no metric result from those shots can be trusted.

`with_square_mm` scales the marker along with the square, because a rescaled print rescales
both — overriding the square alone would describe a board nobody printed.

In [ ]:
MIN_CORNERS = 6          # below this a view cannot pose the board; solvePnP needs 4 and
                         # will happily return nonsense from 4 noisy ones
MAX_INCIDENCE_DEG = 70.  # a board this far from face-on has almost no depth extent left:
                         # corners crowd together and the pose goes soft along the ray


@dataclass
class CharucoSpec:
    """One physical board, in millimetres.

    ``square_mm`` is the *measured* pitch of the print in front of you, not the nominal one.
    ``marker_mm`` must be smaller -- OpenCV insets the marker inside the white square.
    """

    cols: int
    rows: int
    square_mm: float
    marker_mm: float
    dict_name: str = "DICT_4X4_100"
    name: str = ""

    def __post_init__(self):
        if self.marker_mm >= self.square_mm:
            raise ValueError(f"marker {self.marker_mm} >= square {self.square_mm} mm")

    @property
    def board(self):
        """The `cv2.aruco.CharucoBoard`, cached like `rig.Camera.K_inv` is."""
        b = getattr(self, "_board", None)
        if b is None:
            b = cv2.aruco.CharucoBoard(
                (self.cols, self.rows), float(self.square_mm), float(self.marker_mm),
                cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, self.dict_name)))
            object.__setattr__(self, "_board", b)
        return b

    @property
    def detector(self):
        d = getattr(self, "_detector", None)
        if d is None:
            d = cv2.aruco.CharucoDetector(self.board)   # sub-pixel refines internally
            object.__setattr__(self, "_detector", d)
        return d

    @property
    def n_corners(self):
        return (self.cols - 1) * (self.rows - 1)

    @property
    def size_mm(self):
        return (self.cols * self.square_mm, self.rows * self.square_mm)

    @property
    def corners_mm(self):
        """Interior corners in board coordinates, ``(n_corners, 3)``, z = 0."""
        return np.asarray(self.board.getChessboardCorners(), dtype=np.float64)

    def with_square_mm(self, square_mm):
        """Copy at a measured pitch, scaling the marker by the same factor."""
        k = float(square_mm) / self.square_mm
        return CharucoSpec(self.cols, self.rows, float(square_mm), self.marker_mm * k,
                           self.dict_name, self.name)

    def summary(self):
        w, h = self.size_mm
        return {"board": self.name, "squares": [self.cols, self.rows],
                "square_mm": round(self.square_mm, 4), "marker_mm": round(self.marker_mm, 4),
                "dictionary": self.dict_name, "n_corners": self.n_corners,
                "size_mm": [round(w, 3), round(h, 3)]}


# The 6x8 is the legacy board behind vision/board_images/8x6/; its print scale is unverified,
# so it is here for reproducing old runs, not for new ones.
BOARDS = {
    "9x6_letter": CharucoSpec(9, 6, 16.667, 12.5, "DICT_4X4_100", "9x6_letter"),
    "6x8_30mm": CharucoSpec(6, 8, 30.0, 22.5, "DICT_4X4_100", "6x8_30mm"),
}

# ---- SET THIS from calipers on the actual print, then re-run everything below ----
SQUARE_MM = None            # None = nominal; a float overrides it

SPEC = BOARDS["9x6_letter"]
if SQUARE_MM is not None:
    SPEC = SPEC.with_square_mm(SQUARE_MM)
else:
    print("NOTE: nominal square size. Printers rescale; measure the print and set "
          "SQUARE_MM, or every distance this rig reports carries the same error.\n")

for k, v in SPEC.summary().items():
    print(f"{k:12s} {v}")

### 1a. A print whose scale you can check

Lays the board out at true size on a real page and prints a 100 mm ruler bar beside it, from
the same mm-to-px conversion as the board — so if the bar measures 100 mm the squares are the
stated pitch, and if it does not, both are wrong by the same factor.

The pattern is rasterised at a whole number of pixels per square so every square is
identical, which quantises the pitch: 16.667 mm at 600 dpi becomes 394 px = **16.6793 mm**,
0.07 % high. That returned number, not the nominal one, is what was drawn.

In [ ]:
PAGES = {"letter": (215.9, 279.4), "a4": (210.0, 297.0)}


def generate_pdf(spec, path, page="letter", dpi=600, margin_mm=12.0):
    """Printable sheet at true size. Returns ``(path, actual_square_mm)``."""
    page_mm = PAGES[page]
    mm2px = lambda mm: int(round(mm * dpi / 25.4))   # noqa: E731

    square_px = mm2px(spec.square_mm)
    actual_square_mm = square_px * 25.4 / dpi
    board_w, board_h = spec.cols * square_px, spec.rows * square_px
    page_w, page_h = mm2px(page_mm[0]), mm2px(page_mm[1])
    if board_w > page_w - 2 * mm2px(margin_mm) or board_h > page_h - 2 * mm2px(margin_mm):
        raise ValueError(f"board {board_w}x{board_h} px does not fit {page_w}x{page_h} px")

    page_img = np.full((page_h, page_w), 255, np.uint8)
    x0, y0 = (page_w - board_w) // 2, mm2px(margin_mm) + mm2px(14.0)
    page_img[y0:y0 + board_h, x0:x0 + board_w] = spec.board.generateImage(
        (board_w, board_h), marginSize=0)

    s = dpi / 600.0                                   # text metrics tuned at 600 dpi
    def text(msg, x, y, size=1.6, thick=4):
        cv2.putText(page_img, msg, (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX,
                    size * s, 0, max(1, int(round(thick * s))), cv2.LINE_AA)

    text(f"{spec.name}  {spec.cols}x{spec.rows} squares  {actual_square_mm:.4f} mm pitch  "
         f"marker {spec.marker_mm:.3f} mm  {spec.dict_name}", x0, y0 - mm2px(4.0))

    bar_y, bar_len = y0 + board_h + mm2px(18.0), mm2px(100.0)
    bar_x = (page_w - bar_len) // 2
    cv2.line(page_img, (bar_x, bar_y), (bar_x + bar_len, bar_y), 0, max(1, int(6 * s)))
    for mm in range(0, 101, 10):
        tick = mm2px(6.0 if mm % 50 else 10.0)
        x = bar_x + mm2px(float(mm))
        cv2.line(page_img, (x, bar_y), (x, bar_y + tick), 0, max(1, int(6 * s)))
    text("0", bar_x - mm2px(2.0), bar_y + mm2px(18.0), 1.4)
    text("100 mm", bar_x + bar_len - mm2px(12.0), bar_y + mm2px(18.0), 1.4)

    for i, line in enumerate([
            "Print at 100% / actual size -- no 'fit to page', no scaling.",
            "Then measure this bar. If it is not 100.0 mm, the print is scaled:",
            "measure one square with calipers and set SQUARE_MM in this notebook.",
            "Mount on glass or foam board. Paper curl is a systematic bias."]):
        text(line, bar_x, bar_y + mm2px(30.0 + 7.0 * i), 1.15, 3)

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(page_img).save(path, format="PDF", dpi=(dpi, dpi))
    return path, actual_square_mm


# Uncomment to regenerate the print, then measure the bar before shooting anything.
# path, actual = generate_pdf(SPEC, VISION / "charuco_9x6_letter.pdf")
# print(f"wrote {path}\nactual pitch {actual:.4f} mm "
#       f"({(actual / SPEC.square_mm - 1) * 100:+.3f}% vs requested)")

## 2. Detection

`CharucoDetector.detectBoard` sub-pixel refines internally. **OpenCV 5 returns flat
`(N,2)` / `(N,)`** where OpenCV 4 returned `(N,1,2)` / `(N,1)`, and
`interpolateCornersCharuco` / `calibrateCameraCharuco` are gone entirely — so
`writeup/camera_calibration.ipynb`, which uses both, is dead code rather than a reference.
`detect` normalises the shapes once so nothing below reshapes by hand.

`board_incidence_deg` reads 0° face-on and 90° edge-on. It takes the board normal against the
optical axis as an **undirected line** (`abs`), because which face points at the lens is not
what conditions the pose — how obliquely you see it is.

In [ ]:
def detect(spec, image):
    """ChArUco corners in one image: ``(corners (N,2), ids (N,))`` or ``(None, None)``."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    corners, ids, _, _ = spec.detector.detectBoard(gray)
    if corners is None or ids is None or len(corners) == 0:
        return None, None
    return (np.asarray(corners, dtype=np.float64).reshape(-1, 2),
            np.asarray(ids, dtype=np.int32).reshape(-1))


def match_points(spec, corners, ids):
    """``(object (N,1,3), image (N,1,2))`` float32 for cv2.calibrateCamera/stereoCalibrate."""
    obj, img = spec.board.matchImagePoints(
        np.asarray(corners, dtype=np.float32).reshape(-1, 1, 2),
        np.asarray(ids, dtype=np.int32).reshape(-1, 1))
    if obj is None or img is None or len(obj) == 0:
        return None, None
    return (np.asarray(obj, dtype=np.float32).reshape(-1, 1, 3),
            np.asarray(img, dtype=np.float32).reshape(-1, 1, 2))


def solve_board_pose(spec, corners, ids, K, dist):
    """``(rvec, tvec)`` of the board in camera coordinates, mm. ``None`` if unsolvable.

    IPPE is the planar-target solver -- exact for z = 0 object points and better conditioned
    than the iterative default -- then LM refinement minimises the actual reprojection error,
    which is the quantity being gated on.
    """
    if ids is None or len(ids) < MIN_CORNERS:
        return None
    obj, img = match_points(spec, corners, ids)
    if obj is None:
        return None
    ok, rvec, tvec = cv2.solvePnP(obj, img, K, dist, flags=cv2.SOLVEPNP_IPPE)
    if not ok:
        return None
    rvec, tvec = cv2.solvePnPRefineLM(obj, img, K, dist, rvec, tvec)
    return rvec.reshape(3), tvec.reshape(3)


def board_incidence_deg(rvec):
    """Angle from face-on, degrees. 0 = square to the camera, 90 = edge-on."""
    R, _ = cv2.Rodrigues(np.asarray(rvec, dtype=np.float64).reshape(3, 1))
    return math.degrees(math.acos(float(np.clip(abs(R[2, 2]), 0.0, 1.0))))


def pose_matrix(rvec, tvec):
    """``T_cam_board`` (4x4): board coordinates -> camera coordinates."""
    T = np.eye(4)
    T[:3, :3], _ = cv2.Rodrigues(np.asarray(rvec, dtype=np.float64).reshape(3, 1))
    T[:3, 3] = np.asarray(tvec, dtype=np.float64).reshape(3)
    return T


def annotate(image, spec, corners, ids, rvec=None, tvec=None, K=None, dist=None):
    """Copy of ``image`` with detected corners and, if posed, the board axes."""
    out = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR) if image.ndim == 2 else image.copy()
    if ids is not None and len(ids):
        # OpenCV 5 hands back flat arrays; the drawing helpers still want (N,1,*).
        cv2.aruco.drawDetectedCornersCharuco(
            out, np.asarray(corners, dtype=np.float32).reshape(-1, 1, 2),
            np.asarray(ids, dtype=np.int32).reshape(-1, 1))
    if rvec is not None and K is not None:
        cv2.drawFrameAxes(out, K, dist, rvec, tvec, 2.0 * spec.square_mm)
    return out

## 3. The camera: pick the mode before you shoot anything

Measured on the ELP global-shutter module on this bench (VID `0x32E4`, PID `0x9281`, the
OV9281 sensor), by timing real reads — `CAP_PROP_FPS` is only what the driver claims:

| requested | delivered | measured fps | relation to native | field of view |
|---|---|---|---|---|
| 1280×800 | 1280×800 | **119** | **native** | full |
| 640×400 | 640×400 | **217** | **exact 0.5× rescale** | full |
| 640×480 | 640×480 | 107 | crop | narrower |
| 1280×720 | 1280×720 | 53 | crop | narrower |
| 320×240 | 320×240 | 333 | crop | much narrower |

Two results here are worth more than the table.

**1280×720 is the worst mode available.** It is *slower* than the full 1280×800 (53 vs 119
fps) and sees *less*, because 800 is the sensor's native height and 720 is a windowed
readout of it. The reflex to ask for "720p" costs you half the frame rate and part of the
image. Ask for 1280×800.

**Only 640×400 is a true rescale.** Cross-correlating each mode against a resize versus a
centre crop of the native frame: 640×400 matches the *resize* at 0.9994, while 640×480,
1280×720 and 320×240 all match the *crop* (0.995, 0.9996, 0.991). This decides whether
intrinsics transfer:

- 1280×800 → 640×400 is a clean $\times 0.5$, so `rig.Camera.scaled(0.5)` is exactly right —
  $f_x, f_y, c_x, c_y$ all halve.
- Every other mode is a **crop**, where $f_x, f_y$ are *unchanged* and only $c_x, c_y$ shift
  by the discarded margin. Applying a scale factor there is silently wrong, in the same
  quiet way §14.4's print-scale error is wrong: plausible numbers, no warning.

So: **calibrate at 1280×800, then run at 1280×800 (119 fps) or 640×400 (217 fps).** Those
two are one calibration. Anything else needs its own.

Also measured: the sensor is **monochrome** — all three BGR channels come back bit-identical
— so `grayscale=True` throws away nothing and `IMREAD_GRAYSCALE` is the right reader (§13).
And as on every other camera here, macOS refuses exposure control: every
`CAP_PROP_EXPOSURE` / `AUTO_EXPOSURE` set returns `False` and reads back `-1`. Light the
scene rather than fighting the driver.

At 217 fps this is also the first camera on the bench near the 240 fps target the pose
README is written against; `sources.CameraSource`'s drop-oldest grabber matters at that rate
in a way it never did at the C270's 28 fps.

In [ ]:
# ELP OV9281 global shutter, measured on this bench -- see the table above.
NATIVE_W, NATIVE_H = 1280, 800     # native mode: 119 fps, full field of view
FAST_W, FAST_H = 640, 400          # exact 0.5x rescale: 217 fps, same field of view


def list_cameras(max_index=4, backend=None):
    """[(index, 'WxH')] for every camera that will actually stream.

    macOS has to probe -- AVFoundation exposes no device names through OpenCV -- so this
    costs about a second per index. USB cameras enumerate BEFORE the built-in FaceTime, so
    the ELP is normally index 0.
    """
    backend = backend or (cv2.CAP_AVFOUNDATION if sys.platform == "darwin" else cv2.CAP_V4L2)
    found = []
    for i in range(max_index):
        cap = cv2.VideoCapture(i, backend)
        if cap.isOpened() and cap.read()[0]:
            found.append((i, f"{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}"
                             f"x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}"))
        cap.release()
    return found


list_cameras()

## 3a. Capture: stereo pairs

Reuses `sources.open_stereo`, which already wraps two threaded drop-oldest `CameraSource`s
and **measures the capture skew** rather than assuming it. Sync barely matters here — the
board is static when you press SPACE — but `skew_stats()` is logged anyway, because it is the
number that will matter for the live pose loop.

On macOS select by index: AVFoundation exposes no device names through OpenCV, and it
enumerates USB cameras *before* the built-in FaceTime, so the plugged-in pair is usually
0 and 1.

**Shoot at the sensor's native mode — see §3 for why 1280×800 and not 720p.**

**Shoot for coverage, not count.** Sweep the board through position, tilt and roll, fill the
frame corners as well as the centre, and keep it angled to bisect the two cameras so neither
sees it near edge-on. The live overlay prints corner count and incidence for both cameras so
you can reject a useless view before saving it. ~25 good pairs is plenty.

In [ ]:
def capture_pairs(spec, cam_a="camera:0", cam_b="camera:1", out_dir=PAIR_DIR,
                  width=NATIVE_W, height=NATIVE_H, max_skew_s=0.05):
    """Live preview; SPACE saves a pair, q quits. Returns the pairs.csv path.

    Matching index in A/ and B/ is the pairing key -- nothing else pairs them, so never
    delete a file from one side without deleting its partner.
    """
    (out_dir / "A").mkdir(parents=True, exist_ok=True)
    (out_dir / "B").mkdir(parents=True, exist_ok=True)
    rows, n = [], 0

    src = sources.open_stereo([cam_a, cam_b], max_skew_s=max_skew_s,
                              width=width, height=height, grayscale=True)
    try:
        while True:
            item = src.read()
            if item is None:
                print("source ended")
                break
            t, (fa, fb) = item

            panels, info = [], []
            for frame in (fa, fb):
                corners, ids = detect(spec, frame)
                n_det = 0 if ids is None else len(ids)
                inc = float("nan")
                if n_det >= MIN_CORNERS:
                    # Incidence only needs a rough K; the pixel scale cancels in the angle.
                    f = max(frame.shape) * 1.2
                    K = np.array([[f, 0, frame.shape[1] / 2],
                                  [0, f, frame.shape[0] / 2], [0, 0, 1.0]])
                    got = solve_board_pose(spec, corners, ids, K, np.zeros(5))
                    if got is not None:
                        inc = board_incidence_deg(got[0])
                info.append((n_det, inc))
                panels.append(annotate(frame, spec, corners, ids))

            for panel, (n_det, inc), tag in zip(panels, info, "AB"):
                ok = n_det >= MIN_CORNERS and not (inc > MAX_INCIDENCE_DEG)
                cv2.putText(panel, f"{tag}  {n_det}/{spec.n_corners} corners  "
                            f"incidence {inc:5.1f} deg", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                            (0, 200, 0) if ok else (0, 0, 255), 2, cv2.LINE_AA)
            view = np.hstack(panels)
            cv2.putText(view, f"saved {n}   SPACE = save, q = quit",
                        (10, view.shape[0] - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (255, 255, 255), 2, cv2.LINE_AA)
            cv2.imshow("stereo calibration capture", view)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            if key == ord(" "):
                cv2.imwrite(str(out_dir / "A" / f"pair_{n:03d}.png"), fa)
                cv2.imwrite(str(out_dir / "B" / f"pair_{n:03d}.png"), fb)
                rows.append((n, t, info[0][0], info[0][1], info[1][0], info[1][1]))
                print(f"pair {n:03d}: A {info[0][0]:2d} corners @ {info[0][1]:5.1f} deg   "
                      f"B {info[1][0]:2d} corners @ {info[1][1]:5.1f} deg")
                n += 1
    finally:
        cv2.destroyAllWindows()
        stats = src.skew_stats()
        src.close()

    csv_path = out_dir / "pairs.csv"
    with open(csv_path, "w") as fh:
        for k, v in stats.items():
            fh.write(f"# skew_{k}, {v}\n")
        fh.write("index,t_capture,n_corners_a,incidence_a_deg,n_corners_b,incidence_b_deg\n")
        for r in rows:
            fh.write("{0},{1:.6f},{2},{3:.3f},{4},{5:.3f}\n".format(*r))
    print(f"\n{n} pairs -> {out_dir}")
    print(f"capture skew: {stats}")
    return csv_path

In [ ]:
# Live capture. Interactive -- run this cell only when the cameras are plugged in.
# capture_pairs(SPEC, cam_a="camera:0", cam_b="camera:1")

## 3b. Capture: one camera, for intrinsics or a dataset

The same loop as §3a with one camera. Use it for a single camera's intrinsics, or — with
`overlay=False` — as a plain shutter button for collecting robot footage.

**For intrinsics, coverage beats count.** The solver needs the board to reach the frame
*corners*, because that is the only place radial distortion is large enough to measure; a
set shot comfortably in the middle leaves $k_1, k_2, k_3$ fitting noise. It also needs real
tilt — a coplanar target viewed at near-constant angle leaves focal length and distortion
poorly separated, which is exactly the warning §5 prints. The existing
`vision/board_images/9x6/` set spans only 16.5° of orientation spread and is the cautionary
example. Aim for 20–30 frames: centre, each corner, each edge, then tilted 30–50° about both
axes, at two or three distances.

**Lock the focus first.** These M12-lens modules focus by screwing the barrel, and focal
length moves with it — turning the lens after calibrating silently invalidates $f_x, f_y$.
Set focus for the working distance, lock it (a dab of nail varnish on the thread is the
usual bench fix), and only then shoot.

In [ ]:
def capture_photos(out_dir, spec=None, cam="camera:0", width=NATIVE_W, height=NATIVE_H,
                   overlay=True, prefix="img"):
    """Live preview; SPACE saves a frame, q quits. Returns the number saved.

    ``spec`` non-None and ``overlay`` true draws the ChArUco detection and prints corner
    count and incidence, so a useless frame is visible before it is saved rather than at
    calibration time. Pass ``overlay=False`` for plain dataset capture.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    n = len(list(out_dir.glob(f"{prefix}*.png")))    # append, never clobber a set
    if n:
        print(f"{out_dir} already holds {n} frame(s); continuing from there")

    src = sources.open_source(cam, width=width, height=height, grayscale=True)
    try:
        while True:
            item = src.read()
            if item is None:
                print("source ended")
                break
            _, frame = item

            if overlay and spec is not None:
                corners, ids = detect(spec, frame)
                n_det = 0 if ids is None else len(ids)
                inc = float("nan")
                if n_det >= MIN_CORNERS:
                    f = max(frame.shape) * 1.2       # rough K; the angle is scale-free
                    K = np.array([[f, 0, frame.shape[1] / 2],
                                  [0, f, frame.shape[0] / 2], [0, 0, 1.0]])
                    got = solve_board_pose(spec, corners, ids, K, np.zeros(5))
                    if got is not None:
                        inc = board_incidence_deg(got[0])
                view = annotate(frame, spec, corners, ids)
                ok = n_det >= MIN_CORNERS and not (inc > MAX_INCIDENCE_DEG)
                cv2.putText(view, f"{n_det}/{spec.n_corners} corners  "
                            f"incidence {inc:5.1f} deg", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                            (0, 200, 0) if ok else (0, 0, 255), 2, cv2.LINE_AA)
            else:
                view = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR) if frame.ndim == 2 else frame.copy()

            cv2.putText(view, f"saved {n}   SPACE = save, q = quit",
                        (10, view.shape[0] - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (255, 255, 255), 2, cv2.LINE_AA)
            cv2.imshow("capture", view)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            if key == ord(" "):
                # PNG, not JPEG: §13 measured a 0.34 px shift in fx from nothing more than
                # two JPEG grayscale decode paths disagreeing by 2 grey levels. Lossless
                # costs disk and removes the whole question.
                cv2.imwrite(str(out_dir / f"{prefix}_{n:03d}.png"), frame)
                n += 1
    finally:
        cv2.destroyAllWindows()
        src.close()

    print(f"{n} frame(s) in {out_dir}")
    return n

In [ ]:
# Board photos for one camera's intrinsics. Interactive -- run with the camera plugged in.
# capture_photos(OUT_DIR / "intrinsics_A", spec=SPEC, cam="camera:0")

# Plain dataset capture, no board overlay:
# capture_photos(ESP32 / "results" / "robot_dataset", cam="camera:0", overlay=False,
#                prefix="frame")

## 4. Load what was shot, and detect

Pairs are keyed by matching index in `A/` and `B/`. A view is kept only if it clears
`MIN_CORNERS`; incidence is recorded now and enforced later, once real intrinsics exist.

In [ ]:
def load_views(spec, pair_dir=PAIR_DIR, pattern="*.png"):
    """Detect corners in every image. Returns ``(views_a, views_b, image_size)``.

    Each view is a dict with ``index``, ``path``, ``corners``, ``ids``. Indices present on
    only one side are kept -- they still constrain that camera's intrinsics, they just
    cannot contribute to the extrinsic.
    """
    out, size = {}, None
    for tag in "AB":
        views = []
        for path in sorted((Path(pair_dir) / tag).glob(pattern)):
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"  skip {tag}/{path.name}: unreadable")
                continue
            wh = img.shape[1], img.shape[0]
            if size is None:
                size = wh
            elif wh != size:
                raise ValueError(f"{path}: size {wh} != {size}")
            corners, ids = detect(spec, img)
            if ids is None or len(ids) < MIN_CORNERS:
                print(f"  skip {tag}/{path.name}: {0 if ids is None else len(ids)} corners")
                continue
            views.append({"index": path.stem, "path": path, "corners": corners, "ids": ids})
        out[tag] = views
        print(f"camera {tag}: {len(views)} usable views, "
              f"{np.mean([len(v['ids']) for v in views]):.1f} corners mean"
              if views else f"camera {tag}: no usable views")
    return out["A"], out["B"], size

## 5. Per-camera intrinsics

Each camera is solved from **all of its own** usable views, not just the shared ones — a view
that only camera A saw still constrains A's focal length and distortion.

Default 5-coefficient model `(k1,k2,p1,p2,k3)`. `CALIB_RATIONAL_MODEL` adds three more radial
terms but overfits unless many views reach the image border.

A single planar board gives coplanar object points, which leaves intrinsics ill-conditioned
unless the board genuinely tilts and rotates between views — so the orientation spread is
measured and warned about rather than assumed.

In [ ]:
MIN_VIEWS = 8          # 4 is the algebraic minimum; 4 is not a calibration
MIN_ORIENT_SPREAD_DEG = 20.0


def calibrate_intrinsics(spec, views, image_size, name=""):
    """``(K, dist, info)`` from one camera's views via cv2.calibrateCamera."""
    obj_pts, img_pts = [], []
    for v in views:
        o, i = match_points(spec, v["corners"], v["ids"])
        if o is not None:
            obj_pts.append(o)
            img_pts.append(i)
    if len(obj_pts) < MIN_VIEWS:
        raise RuntimeError(f"camera {name}: {len(obj_pts)} usable views, need {MIN_VIEWS}")

    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(
        obj_pts, img_pts, image_size, None, None)
    dist = np.asarray(dist, dtype=np.float64).ravel()

    per_view, incidences = [], []
    for o, i, rv, tv in zip(obj_pts, img_pts, rvecs, tvecs):
        proj, _ = cv2.projectPoints(o, rv, tv, K, dist)
        err = proj.reshape(-1, 2) - i.reshape(-1, 2)
        per_view.append(float(np.sqrt(np.mean(np.sum(err ** 2, axis=1)))))
        incidences.append(board_incidence_deg(rv))

    spread = float(np.std(incidences))
    info = {"n_views": len(obj_pts), "rms_px": float(rms), "per_view_rms_px": per_view,
            "incidence_deg": incidences, "orientation_spread_deg": spread,
            "image_size": list(image_size)}

    print(f"camera {name}: {len(obj_pts)} views, RMS {rms:.4f} px, "
          f"worst view {max(per_view):.4f} px")
    print(f"  fx={K[0,0]:.2f} fy={K[1,1]:.2f} cx={K[0,2]:.2f} cy={K[1,2]:.2f}")
    print(f"  dist {np.array2string(dist, precision=5)}")
    print(f"  incidence {min(incidences):.1f}-{max(incidences):.1f} deg, "
          f"spread {spread:.1f} deg")
    if spread < MIN_ORIENT_SPREAD_DEG:
        print(f"  WARNING: orientation spread {spread:.1f} deg is under "
              f"{MIN_ORIENT_SPREAD_DEG}. Coplanar points at near-identical angles leave "
              f"focal length and distortion poorly separated. Tilt the board more.")
    return K, dist, info


def intrinsics_from_dir(img_dir, spec=SPEC, pattern="*.png", name=None, decode="gray"):
    """``(K, dist, info)`` for one camera from a directory of board photos.

    The single-camera path: point it at what §3b captured. ``decode`` selects the JPEG
    grayscale reader, which is not cosmetic -- see §13, where the two paths move $f_x$ by
    0.34 px. ``"gray"`` matches how `sources.ImageSource` feeds the live pipeline.
    """
    img_dir = Path(img_dir)
    views, size = [], None
    for path in sorted(img_dir.glob(pattern)):
        if decode == "gray":
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        else:
            colour = cv2.imread(str(path))
            img = None if colour is None else cv2.cvtColor(colour, cv2.COLOR_BGR2GRAY)
        if img is None:
            continue
        wh = (img.shape[1], img.shape[0])
        if size is None:
            size = wh
        elif wh != size:
            raise ValueError(
                f"{path.name}: {wh} != {size}. Intrinsics are per-mode, and on this sensor "
                f"most modes are crops rather than rescales (§3), so a mixed-resolution "
                f"set cannot be calibrated together.")
        corners, ids = detect(spec, img)
        if ids is not None and len(ids) >= MIN_CORNERS:
            views.append({"index": path.stem, "corners": corners, "ids": ids})
    if not views:
        raise FileNotFoundError(f"no usable board photos matching {pattern} in {img_dir}")
    return calibrate_intrinsics(spec, views, size, name or img_dir.name)


# One camera, start to finish:
#   capture_photos(OUT_DIR / "intrinsics_A", spec=SPEC, cam="camera:0")   # from 3b
#   K, dist, info = intrinsics_from_dir(OUT_DIR / "intrinsics_A")
#   save_intrinsics(OUT_DIR / "camera_intrinsics_A.npz", K, dist, SPEC, info)  # from 10

## 6. Extrinsics

Two stages, because they fail differently.

**Seed — per-pair, closed form.** For each pair, `solvePnP` in both cameras and compose
$T_{B\leftarrow A} = T_{B\leftarrow\text{board}} \, T_{A\leftarrow\text{board}}^{-1}$.
Rotations average with `Rotation.mean()` — the proper chordal mean on SO(3), not an
elementwise matrix average, which is not a rotation. Translation uses the **median**, which
survives one bad pair.

The **spread across pairs is the honest uncertainty**, and it is the number a bundle RMS
hides: `stereoCalibrate` can report 0.3 px while one pair disagrees by 5°.

**Refine — `cv2.stereoCalibrate` with `CALIB_FIX_INTRINSIC`**, seeded from the median.
Intrinsics stay fixed so their error cannot leak into the extrinsic.

The step that is easy to miss: **intersect the ChArUco ids per pair.** Each view detects a
different corner subset, and `stereoCalibrate` requires the *same* object points in both
lists. Feeding it unintersected lists is silently wrong whenever the two cameras saw
different corners — which is always.

OpenCV's `R, T` map camera A into camera B ($X_B = R X_A + T$), so they *are*
$T_{B\leftarrow A}$.

In [ ]:
MIN_COMMON_CORNERS = 8


def pair_views(spec, views_a, views_b, K_a, dist_a, K_b, dist_b,
               max_incidence_deg=MAX_INCIDENCE_DEG):
    """Pairs sharing enough corners at usable incidence. Returns a list of dicts."""
    by_b = {v["index"]: v for v in views_b}
    pairs, rejected = [], []

    for va in views_a:
        vb = by_b.get(va["index"])
        if vb is None:
            continue
        common = np.intersect1d(va["ids"], vb["ids"])
        if len(common) < MIN_COMMON_CORNERS:
            rejected.append((va["index"], f"{len(common)} common corners"))
            continue

        sel_a = np.isin(va["ids"], common)
        sel_b = np.isin(vb["ids"], common)
        # intersect1d returns sorted ids, and isin preserves each view's own order, so
        # re-sort both by id to guarantee row i is the same physical corner in both.
        ord_a = np.argsort(va["ids"][sel_a])
        ord_b = np.argsort(vb["ids"][sel_b])
        ids = va["ids"][sel_a][ord_a]
        img_a = va["corners"][sel_a][ord_a]
        img_b = vb["corners"][sel_b][ord_b]
        assert np.array_equal(ids, vb["ids"][sel_b][ord_b])

        pose_a = solve_board_pose(spec, img_a, ids, K_a, dist_a)
        pose_b = solve_board_pose(spec, img_b, ids, K_b, dist_b)
        if pose_a is None or pose_b is None:
            rejected.append((va["index"], "solvePnP failed"))
            continue
        inc_a, inc_b = board_incidence_deg(pose_a[0]), board_incidence_deg(pose_b[0])
        if max(inc_a, inc_b) > max_incidence_deg:
            rejected.append((va["index"], f"incidence {inc_a:.0f}/{inc_b:.0f} deg"))
            continue

        pairs.append({"index": va["index"], "ids": ids,
                      "obj": spec.corners_mm[ids].astype(np.float32).reshape(-1, 1, 3),
                      "img_a": img_a.astype(np.float32).reshape(-1, 1, 2),
                      "img_b": img_b.astype(np.float32).reshape(-1, 1, 2),
                      "T_a": pose_matrix(*pose_a), "T_b": pose_matrix(*pose_b),
                      "incidence_a": inc_a, "incidence_b": inc_b})

    print(f"{len(pairs)} usable pairs, {len(rejected)} rejected")
    for idx, why in rejected:
        print(f"  reject {idx}: {why}")
    return pairs


def seed_extrinsic(pairs):
    """Median ``T_camB_camA`` over pairs, with the spread that says whether to trust it."""
    Ts = np.array([p["T_b"] @ np.linalg.inv(p["T_a"]) for p in pairs])
    rots = Rotation.from_matrix(Ts[:, :3, :3])
    mean_rot = rots.mean()
    t_med = np.median(Ts[:, :3, 3], axis=0)

    T = np.eye(4)
    T[:3, :3] = mean_rot.as_matrix()
    T[:3, 3] = t_med

    ang = np.degrees((rots * mean_rot.inv()).magnitude())
    lin = np.linalg.norm(Ts[:, :3, 3] - t_med, axis=1)
    spread = {"rot_deg_median": float(np.median(ang)), "rot_deg_max": float(ang.max()),
              "trans_mm_median": float(np.median(lin)), "trans_mm_max": float(lin.max()),
              "per_pair_rot_deg": ang.tolist(), "per_pair_trans_mm": lin.tolist()}
    print(f"seed from {len(pairs)} pairs: baseline {np.linalg.norm(t_med):.2f} mm")
    print(f"  pair-to-pair spread: rotation {spread['rot_deg_median']:.3f} deg median / "
          f"{spread['rot_deg_max']:.3f} worst, translation "
          f"{spread['trans_mm_median']:.3f} mm median / {spread['trans_mm_max']:.3f} worst")
    worst = int(np.argmax(ang))
    if ang.max() > 5 * max(np.median(ang), 1e-6):
        print(f"  pair {pairs[worst]['index']} is the outlier at {ang.max():.2f} deg -- "
              f"look at that image before trusting the bundle")
    return T, spread


def refine_extrinsic(pairs, K_a, dist_a, K_b, dist_b, image_size, seed_T,
                     fix_intrinsic=True):
    """cv2.stereoCalibrate over all pairs. Returns ``(T_camB_camA, info)``.

    **OpenCV 5 rejects ``CALIB_USE_EXTRINSIC_GUESS`` here** ("stereoCalibrate does not
    support CALIB_USE_EXTRINSIC_GUESS"), so the bundle builds its own initial estimate and
    ``seed_T`` cannot be fed in.  That turns out to be worth more than the seeding would
    have been: the closed-form seed is now a genuinely *independent* estimate, and the
    agreement between it and the bundle is a check rather than a tautology.  The gap is
    reported below; a bundle that walks far from the seed means one of them is wrong.
    """
    flags = cv2.CALIB_FIX_INTRINSIC if fix_intrinsic else 0
    out = cv2.stereoCalibrate(
        [p["obj"] for p in pairs], [p["img_a"] for p in pairs], [p["img_b"] for p in pairs],
        K_a, dist_a, K_b, dist_b, image_size,
        seed_T[:3, :3].copy(), seed_T[:3, 3].copy().reshape(3, 1),
        # perViewErrors as a keyword is what selects the 10-return overload; passing R and
        # T positionally alone is ambiguous and OpenCV picks the 9-return one.
        perViewErrors=np.zeros((len(pairs), 2)),
        flags=flags,
        criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 200, 1e-10))
    rms, K_a, dist_a, K_b, dist_b, R, T, E, F, per_view = out

    T_ba = np.eye(4)
    T_ba[:3, :3], T_ba[:3, 3] = R, np.asarray(T).reshape(3)
    per_view = np.asarray(per_view, dtype=np.float64).reshape(-1, 2)

    gap_deg = float(np.degrees(
        Rotation.from_matrix(T_ba[:3, :3] @ seed_T[:3, :3].T).magnitude()))
    gap_mm = float(np.linalg.norm(T_ba[:3, 3] - seed_T[:3, 3]))
    info = {"rms_px": float(rms), "n_pairs": len(pairs),
            "per_pair_rms_px": per_view.tolist(), "fix_intrinsic": bool(fix_intrinsic),
            "seed_gap_deg": gap_deg, "seed_gap_mm": gap_mm}
    print(f"stereoCalibrate: RMS {rms:.4f} px over {len(pairs)} pairs "
          f"(worst pair {per_view.max():.4f} px)")
    print(f"  baseline {np.linalg.norm(T_ba[:3, 3]):.3f} mm")
    print(f"  agreement with the independent closed-form seed: "
          f"{gap_deg:.4f} deg, {gap_mm:.4f} mm")
    return T_ba, info

## 7. Acceptance

From `docs/pose_localization_project_context.md` §6: RMS **< 0.5 px**, and residuals
**isotropic and structureless**. The second half is the one that matters — a radial or
edge-worse pattern means underfit distortion, i.e. systematic error hiding under a
respectable average.

**The residuals have to come from the joint fit.** Re-solving each camera's board pose
independently measures only that camera's intrinsics; the extrinsic never enters, so a
completely wrong $T_{B\leftarrow A}$ would still produce a clean residual and pass. So
`stereo_residuals` fits **one** board pose per pair that has to explain both views *through*
$T_{B\leftarrow A}$ — camera B's residual then carries the extrinsic error, which is the
thing being gated.

Then bin those residuals by radius from the principal point and compare outer to inner RMS,
and compare the x and y spreads. The gate also checks that the bundle agrees with the
closed-form seed: because OpenCV 5 will not accept an extrinsic guess, the two are genuinely
independent estimates, and their agreement is information rather than a tautology.

In [ ]:
MAX_RMS_PX = 0.5
MAX_RADIAL_RATIO = 1.5


def stereo_residuals(pairs, K_a, dist_a, K_b, dist_b, T_ba):
    """Residuals from the **joint** fit: one board pose per pair, explaining both views.

    Re-solving each camera independently would measure only that camera's intrinsics -- the
    extrinsic never enters, so a completely wrong ``T_ba`` would still produce a beautifully
    clean residual and sail through the gate.  Here a single board pose has to explain both
    views *through* ``T_ba``, so camera B's residual carries the extrinsic error, which is
    the thing actually being tested.
    """
    out = {"A": {"rad": [], "res": []}, "B": {"rad": [], "res": []}, "per_pair": []}
    for p in pairs:
        obj = p["obj"].reshape(-1, 3).astype(np.float64)
        ia = p["img_a"].reshape(-1, 2).astype(np.float64)
        ib = p["img_b"].reshape(-1, 2).astype(np.float64)

        def split(x):
            T_a = pose_matrix(x[:3], x[3:])
            T_b = T_ba @ T_a
            pa, _ = cv2.projectPoints(obj, x[:3], x[3:], K_a, dist_a)
            rb, _ = cv2.Rodrigues(T_b[:3, :3])
            pb, _ = cv2.projectPoints(obj, rb, T_b[:3, 3], K_b, dist_b)
            return pa.reshape(-1, 2) - ia, pb.reshape(-1, 2) - ib

        r0 = np.concatenate([cv2.Rodrigues(p["T_a"][:3, :3])[0].ravel(), p["T_a"][:3, 3]])
        sol = least_squares(lambda x: np.concatenate([r.ravel() for r in split(x)]),
                            r0, method="lm")
        da, db = split(sol.x)

        out["A"]["res"].append(da)
        out["B"]["res"].append(db)
        out["A"]["rad"].append(np.linalg.norm(ia - [K_a[0, 2], K_a[1, 2]], axis=1))
        out["B"]["rad"].append(np.linalg.norm(ib - [K_b[0, 2], K_b[1, 2]], axis=1))
        out["per_pair"].append([float(np.sqrt(np.mean(np.sum(d ** 2, axis=1))))
                                for d in (da, db)])

    for tag in "AB":
        out[tag]["res"] = np.concatenate(out[tag]["res"])
        out[tag]["rad"] = np.concatenate(out[tag]["rad"])
    out["per_pair"] = np.asarray(out["per_pair"])
    out["rms_px"] = float(np.sqrt(np.mean(np.sum(
        np.vstack([out["A"]["res"], out["B"]["res"]]) ** 2, axis=1))))
    print(f"joint stereo residual: {out['rms_px']:.4f} px RMS over both views "
          f"(A {np.sqrt(np.mean(np.sum(out['A']['res'] ** 2, axis=1))):.4f}, "
          f"B {np.sqrt(np.mean(np.sum(out['B']['res'] ** 2, axis=1))):.4f})")
    return out


def structure_report(radii, res, n_bins=4):
    """Radial trend and isotropy of one camera's residuals."""
    edges = np.quantile(radii, np.linspace(0, 1, n_bins + 1))
    binned = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (radii >= lo) & (radii <= hi)
        binned.append(float(np.sqrt(np.mean(np.sum(res[m] ** 2, axis=1)))) if m.any()
                      else float("nan"))
    ratio = binned[-1] / binned[0] if binned[0] > 0 else float("inf")
    return {"rms_px": float(np.sqrt(np.mean(np.sum(res ** 2, axis=1)))),
            "radial_bins_px": binned, "radial_ratio": float(ratio),
            "anisotropy": float(np.std(res[:, 0]) / max(np.std(res[:, 1]), 1e-12)),
            "n_points": int(len(res))}


def acceptance(stereo_info, resid, intr_a, intr_b, struct_a, struct_b, spread):
    """Print the gate and return True only if everything passes."""
    checks = [
        ("joint stereo RMS < 0.5 px", resid["rms_px"] < MAX_RMS_PX,
         f"{resid['rms_px']:.4f} px"),
        ("bundle RMS < 0.5 px", stereo_info["rms_px"] < MAX_RMS_PX,
         f"{stereo_info['rms_px']:.4f} px"),
        ("bundle agrees with closed-form seed", stereo_info["seed_gap_deg"] < 1.0,
         f"{stereo_info['seed_gap_deg']:.3f} deg, {stereo_info['seed_gap_mm']:.3f} mm"),
        ("camera A intrinsics RMS < 0.5 px", intr_a["rms_px"] < MAX_RMS_PX,
         f"{intr_a['rms_px']:.4f} px"),
        ("camera B intrinsics RMS < 0.5 px", intr_b["rms_px"] < MAX_RMS_PX,
         f"{intr_b['rms_px']:.4f} px"),
        ("A residuals structureless", struct_a["radial_ratio"] < MAX_RADIAL_RATIO,
         f"outer/inner {struct_a['radial_ratio']:.2f}"),
        ("B residuals structureless", struct_b["radial_ratio"] < MAX_RADIAL_RATIO,
         f"outer/inner {struct_b['radial_ratio']:.2f}"),
        ("A residuals isotropic", 0.67 < struct_a["anisotropy"] < 1.5,
         f"sx/sy {struct_a['anisotropy']:.2f}"),
        ("B residuals isotropic", 0.67 < struct_b["anisotropy"] < 1.5,
         f"sx/sy {struct_b['anisotropy']:.2f}"),
        ("pairs agree on rotation", spread["rot_deg_max"] < 2.0,
         f"worst {spread['rot_deg_max']:.2f} deg"),
        ("enough pairs", stereo_info["n_pairs"] >= MIN_VIEWS,
         f"{stereo_info['n_pairs']} pairs"),
    ]
    print("acceptance (docs section 6)")
    for label, ok, detail in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label:36s} {detail}")
    passed = all(ok for _, ok, _ in checks)
    print(f"\n{'PASS' if passed else 'FAIL'} overall")
    if not passed:
        print("A failed gate is not a formality: a bad extrinsic produces poses that are\n"
              "smooth, plausible, self-consistent and wrong. Fix the capture, not the limit.")
    return passed

## 8. Run it

Loads the pairs, solves both cameras, solves the extrinsic, and gates the result. Nothing is
written to disk until section 10.

In [ ]:
def run_calibration(spec, pair_dir=PAIR_DIR):
    """Images on disk -> a gated result. Everything downstream reads the returned dict."""
    views_a, views_b, image_size = load_views(spec, pair_dir)
    print(f"image size {image_size}\n")

    K_a, dist_a, intr_a = calibrate_intrinsics(spec, views_a, image_size, "A")
    print()
    K_b, dist_b, intr_b = calibrate_intrinsics(spec, views_b, image_size, "B")
    print()

    pairs = pair_views(spec, views_a, views_b, K_a, dist_a, K_b, dist_b)
    if not pairs:
        raise RuntimeError("no usable pairs -- check the rejection reasons above")
    seed_T, spread = seed_extrinsic(pairs)
    T_ba, stereo_info = refine_extrinsic(pairs, K_a, dist_a, K_b, dist_b, image_size, seed_T)
    print()

    resid = stereo_residuals(pairs, K_a, dist_a, K_b, dist_b, T_ba)
    struct_a = structure_report(resid["A"]["rad"], resid["A"]["res"])
    struct_b = structure_report(resid["B"]["rad"], resid["B"]["res"])
    print()
    passed = acceptance(stereo_info, resid, intr_a, intr_b, struct_a, struct_b, spread)

    return {"pairs": pairs, "image_size": image_size, "K_a": K_a, "dist_a": dist_a,
            "K_b": K_b, "dist_b": dist_b, "intr_a": intr_a, "intr_b": intr_b,
            "T_ba": T_ba, "seed_T": seed_T, "spread": spread, "resid": resid,
            "stereo_info": stereo_info, "struct_a": struct_a, "struct_b": struct_b,
            "passed": passed}


HAVE_PAIRS = (PAIR_DIR / "A").is_dir() and any((PAIR_DIR / "A").glob("*.png"))
CAL = run_calibration(SPEC) if HAVE_PAIRS else None
if CAL is None:
    print(f"no pairs under {PAIR_DIR} -- run section 3a to capture some.")
    print("Sections 9-11 need them. Sections 12-13 do not: the self-test and the")
    print("intrinsics regression run with no hardware and no capture.")

## 9. Diagnostics

Five panels: per-pair reprojection, the residual scatter that the isotropy check summarises,
corner coverage per camera (holes are where distortion is unconstrained), per-pair extrinsic
spread, and the incidence distribution — the last being the data that answers open item 3 in
`docs/pose_localization_project_context.md`, whether one angled board can serve a widely
separated pair or whether it takes a cube.

In [ ]:
def figures(pairs, image_size, resid, stereo_info, spread, out_dir=OUT_DIR):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rad_a, res_a = resid["A"]["rad"], resid["A"]["res"]
    rad_b, res_b = resid["B"]["rad"], resid["B"]["res"]
    per_pair = resid["per_pair"]

    fig, ax = plt.subplots(2, 3, figsize=(16, 9))

    x = np.arange(len(pairs))
    ax[0, 0].bar(x - 0.2, per_pair[:, 0], 0.4, label="A", color="#2a78d6")
    ax[0, 0].bar(x + 0.2, per_pair[:, 1], 0.4, label="B", color="#1baf7a")
    ax[0, 0].axhline(MAX_RMS_PX, color="#e34948", ls="--", label=f"{MAX_RMS_PX} px limit")
    ax[0, 0].set_xticks(x)
    ax[0, 0].set_xticklabels([p["index"] for p in pairs], rotation=90, fontsize=7)
    ax[0, 0].set_ylabel("joint reprojection RMS (px)")
    ax[0, 0].set_title(f"per pair (overall {resid['rms_px']:.3f} px)")

    for a, r, c, name in ((ax[0, 1], res_a, "#2a78d6", "A"), (ax[0, 1], res_b, "#1baf7a", "B")):
        a.scatter(r[:, 0], r[:, 1], s=4, alpha=0.35, color=c, label=name)
    ax[0, 1].axhline(0, lw=0.5, color="k")
    ax[0, 1].axvline(0, lw=0.5, color="k")
    ax[0, 1].set_aspect("equal")
    ax[0, 1].set_xlabel("dx (px)")
    ax[0, 1].set_ylabel("dy (px)")
    ax[0, 1].set_title("residuals - want a round, centred blob")

    for a, rad, res, c, name in ((ax[0, 2], rad_a, res_a, "#2a78d6", "A"),
                                 (ax[0, 2], rad_b, res_b, "#1baf7a", "B")):
        order = np.argsort(rad)
        mag = np.linalg.norm(res, axis=1)[order]
        k = max(1, len(mag) // 30)
        a.plot(rad[order][::k], np.convolve(mag, np.ones(k) / k, "same")[::k],
               color=c, label=name)
    ax[0, 2].set_xlabel("radius from principal point (px)")
    ax[0, 2].set_ylabel("|residual| (px)")
    ax[0, 2].set_title("radial trend - a rising line is underfit distortion")

    for a, views, c, name in ((ax[1, 0], "img_a", "#2a78d6", "A"),
                              (ax[1, 0], "img_b", "#1baf7a", "B")):
        pts = np.concatenate([p[views].reshape(-1, 2) for p in pairs])
        a.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.4, color=c, label=name)
    ax[1, 0].set_xlim(0, image_size[0])
    ax[1, 0].set_ylim(image_size[1], 0)
    ax[1, 0].set_aspect("equal")
    ax[1, 0].set_title("corner coverage - holes are unconstrained distortion")

    ax[1, 1].bar(x - 0.2, spread["per_pair_rot_deg"], 0.4, label="rotation (deg)",
                 color="#eda100")
    ax[1, 1].bar(x + 0.2, spread["per_pair_trans_mm"], 0.4, label="translation (mm)",
                 color="#e34948")
    ax[1, 1].set_xticks(x)
    ax[1, 1].set_xticklabels([p["index"] for p in pairs], rotation=90, fontsize=7)
    ax[1, 1].set_title("per-pair deviation from the median extrinsic")

    ax[1, 2].hist([p["incidence_a"] for p in pairs], bins=12, alpha=0.6,
                  label="A", color="#2a78d6")
    ax[1, 2].hist([p["incidence_b"] for p in pairs], bins=12, alpha=0.6,
                  label="B", color="#1baf7a")
    ax[1, 2].axvline(MAX_INCIDENCE_DEG, color="#e34948", ls="--", label="reject limit")
    ax[1, 2].set_xlabel("board incidence (deg, 0 = face-on)")
    ax[1, 2].set_title("incidence - can one board serve both cameras?")

    for a in ax.ravel():
        a.grid(alpha=0.3)
        a.legend(fontsize=8)
    fig.suptitle(f"stereo calibration - {len(pairs)} pairs at "
                 f"{image_size[0]}x{image_size[1]}")
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    path = out_dir / "stereo_calibration.png"
    fig.savefig(path, dpi=140)
    print(f"figures -> {path}")
    return fig


if CAL:
    figures(CAL["pairs"], CAL["image_size"], CAL["resid"], CAL["stereo_info"],
            CAL["spread"])
else:
    print("no calibration -- capture pairs first")

## 10. Write `stereo_rig.json`

World is camera A, so `T_world_camA = I` and `T_world_camB = (T_{B\leftarrow A})^{-1}`.

Also writes each camera's intrinsics as npz in the richer schema already used by
`vision/camera_calibrations/camera_intrinsics_6x8.npz`, so the provenance travels with the
numbers.

In [ ]:
def build_rig(K_a, dist_a, K_b, dist_b, T_ba, spec, image_size,
              intr_a, intr_b, stereo_info, spread):
    """StereoRig in the camera-A world frame."""
    return StereoRig(
        cameras=(
            Camera(K=K_a, dist=dist_a, T_world_cam=np.eye(4), name="A"),
            Camera(K=K_b, dist=dist_b, T_world_cam=np.linalg.inv(T_ba), name="B"),
        ),
        meta={
            "source": "stereo_calibration.ipynb",
            "world_frame": "camera_A",
            "world_frame_note": (
                "T_world_camA = I. +z is camera A's optical axis, NOT up. baseline_mm and "
                "axis_separation_deg are frame-independent and valid; tilt_seen_deg and any "
                "elevation reading are not, until the rig is rebased onto the drone disk "
                "frame at integration with the pose estimator."),
            "square_mm_measured": SQUARE_MM is not None,
            **spec.summary(),
            "image_size": list(image_size),
            "n_pairs": stereo_info["n_pairs"],
            "rms_px": stereo_info["rms_px"],
            "rms_px_intrinsics": {"A": intr_a["rms_px"], "B": intr_b["rms_px"]},
            "pair_spread": {k: v for k, v in spread.items() if not k.startswith("per_pair")},
            "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        },
    )


def save_intrinsics(path, K, dist, spec, info):
    np.savez(path, camera_matrix=K, dist_coeffs=dist,
             image_size=np.array(info["image_size"]), rms=info["rms_px"],
             board=spec.name, board_squares=np.array([spec.cols, spec.rows]),
             square_len=spec.square_mm, marker_len=spec.marker_mm)
    return path


if CAL is None:
    print("no calibration -- capture pairs first")
elif not CAL["passed"]:
    print("acceptance failed -- not writing a rig. Fix the capture, not the limit.")
else:
    the_rig = build_rig(CAL["K_a"], CAL["dist_a"], CAL["K_b"], CAL["dist_b"], CAL["T_ba"],
                        SPEC, CAL["image_size"], CAL["intr_a"], CAL["intr_b"],
                        CAL["stereo_info"], CAL["spread"])
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"wrote {the_rig.save(RIG_PATH)}")
    for tag in "AB":
        print("wrote", save_intrinsics(OUT_DIR / f"camera_intrinsics_{tag}.npz",
                                       CAL[f"K_{tag.lower()}"], CAL[f"dist_{tag.lower()}"],
                                       SPEC, CAL[f"intr_{tag.lower()}"]))

    print("\nrig geometry (frame-independent quantities only):")
    print(f"  {'baseline_mm':24s} {the_rig.baseline_mm():.3f}")
    print(f"  {'axis_separation_deg':24s} {the_rig.axis_separation_deg():.3f}")
    if the_rig.axis_separation_deg() < 20.0:
        print("  WARNING: under 20 deg between the optical axes. Fusion needs the cameras "
              "to disagree about which direction is depth; below ~20 deg they barely do, "
              "and the fused answer will be little better than one camera.")
    print("\nStill owed, and no code here can do it: triangulate a caliper-measured "
          "distance in the working volume and compare. It uses none of the machinery "
          "above, which is exactly what makes it a check.")

## 11. Scale check on held-out pairs

Triangulates corner pairs and compares against the board's own geometry. Undistort to
normalised coordinates, then $P_A = [I|0]$ and $P_B = [R|t]$, so the triangulated points come
out in camera A's frame in millimetres.

**What this does and does not prove.** It confirms the extrinsic and the *relative* scale are
consistent — a wrong baseline shows up immediately. It shares the board's absolute scale, so
a mis-scaled print passes it happily. Only the caliper measurement of the print closes that,
which is why section 1 keeps insisting on it.

In [ ]:
def scale_check(pairs, K_a, dist_a, K_b, dist_b, T_ba, spec, holdout=None):
    """Triangulated corner distances vs. the board's known geometry, in mm."""
    use = pairs if holdout is None else pairs[-holdout:]
    P_a = np.hstack([np.eye(3), np.zeros((3, 1))])
    P_b = np.hstack([T_ba[:3, :3], T_ba[:3, 3].reshape(3, 1)])

    errs, truths = [], []
    for p in use:
        na = cv2.undistortPoints(p["img_a"], K_a, dist_a).reshape(-1, 2).T
        nb = cv2.undistortPoints(p["img_b"], K_b, dist_b).reshape(-1, 2).T
        X = cv2.triangulatePoints(P_a, P_b, na, nb)
        X = (X[:3] / X[3]).T                                  # (N,3) mm in camera A

        truth = spec.corners_mm[p["ids"]]
        # Every corner pair in the view, so the comparison spans short and long baselines
        # rather than one convenient distance.
        i, j = np.triu_indices(len(X), k=1)
        d_meas = np.linalg.norm(X[i] - X[j], axis=1)
        d_true = np.linalg.norm(truth[i] - truth[j], axis=1)
        errs.append(d_meas - d_true)
        truths.append(d_true)

    errs, truths = np.concatenate(errs), np.concatenate(truths)
    rel = errs / truths
    print(f"scale check over {len(use)} pair(s), {len(errs)} corner-to-corner distances")
    print(f"  absolute error  {np.mean(errs):+.4f} mm mean, "
          f"{np.percentile(np.abs(errs), 95):.4f} mm p95, {np.abs(errs).max():.4f} mm worst")
    print(f"  relative error  {np.mean(rel) * 100:+.4f}% mean, "
          f"{np.percentile(np.abs(rel), 95) * 100:.4f}% p95")
    print("  (shares the board's absolute scale -- a mis-scaled print passes this)")
    return errs, rel


if CAL:
    scale_check(CAL["pairs"], CAL["K_a"], CAL["dist_a"], CAL["K_b"], CAL["dist_b"],
                CAL["T_ba"], SPEC)
else:
    print("no calibration -- capture pairs first")

## 12. Self-test — synthetic round trip

No hardware, no board, no printer. Builds a rig with a **known** extrinsic, projects the
board's corners into both cameras at many synthetic poses, and runs the identical solve. If
this fails, the maths above is wrong and nothing measured with it means anything.

Three cases: exact recovery, behaviour under 0.2 px corner noise, and the id-intersection
path against deliberately disjoint corner subsets.

In [ ]:
def _synthetic_pairs(spec, rig_true, n=18, seed=0, noise_px=0.0, drop=0.0, jitter_deg=18.0):
    """Project the board at n poses into both cameras of a known rig.

    The board normal is placed on the **bisector of the two optical axes**, jittered by
    ``jitter_deg``.  That is exactly the capture this notebook tells you to do -- angle the
    board so neither camera sees it near edge-on -- so the test exercises the recommended
    procedure rather than a geometry nobody would shoot.  Orienting it randomly instead
    leaves camera B at 85-89 degrees incidence on a 60-degree rig, and most pairs get
    thrown out before the solver ever sees them.
    """
    rng = np.random.default_rng(seed)
    K_a, K_b = rig_true.a.K, rig_true.b.K
    dist = np.zeros(5)
    obj = spec.corners_mm
    centre = obj.mean(axis=0)

    bisector = rig_true.a.optical_axis + rig_true.b.optical_axis
    bisector = bisector / np.linalg.norm(bisector)
    R_face = frame_from_normal(bisector)      # board +z along the bisector

    views_a, views_b = [], []
    for k in range(n):
        R_board = (Rotation.from_euler("xyz", rng.uniform(-jitter_deg, jitter_deg, 3),
                                       degrees=True).as_matrix() @ R_face)
        T_world_board = np.eye(4)
        T_world_board[:3, :3] = R_board
        # Put the board's centre near the point both cameras are aimed at.
        T_world_board[:3, 3] = rng.uniform(-40, 40, 3) - R_board @ centre

        T_a_board = np.linalg.inv(rig_true.a.T_world_cam) @ T_world_board
        T_b_board = np.linalg.inv(rig_true.b.T_world_cam) @ T_world_board

        view = {}
        for tag, K, T in (("a", K_a, T_a_board), ("b", K_b, T_b_board)):
            rvec, _ = cv2.Rodrigues(T[:3, :3])
            pts, _ = cv2.projectPoints(obj, rvec, T[:3, 3], K, dist)
            pts = pts.reshape(-1, 2)
            if noise_px:
                pts = pts + rng.normal(0, noise_px, pts.shape)
            ids = np.arange(len(obj), dtype=np.int32)
            if drop:
                keep = rng.random(len(ids)) > drop
                if keep.sum() < MIN_COMMON_CORNERS + 4:
                    keep[:] = True
                pts, ids = pts[keep], ids[keep]
            view[tag] = {"index": f"s{k:03d}", "path": None, "corners": pts, "ids": ids}
        views_a.append(view["a"])
        views_b.append(view["b"])
    T_ba_true = np.linalg.inv(rig_true.b.T_world_cam) @ rig_true.a.T_world_cam
    return views_a, views_b, T_ba_true


def _recover(spec, rig_true, **kw):
    views_a, views_b, T_true = _synthetic_pairs(spec, rig_true, **kw)
    K_a, K_b = rig_true.a.K, rig_true.b.K
    d = np.zeros(5)
    pairs = pair_views(spec, views_a, views_b, K_a, d, K_b, d)
    seed_T, spread = seed_extrinsic(pairs)
    T_ba, info = refine_extrinsic(pairs, K_a, d, K_b, d, (960, 720), seed_T)
    ang = np.degrees(Rotation.from_matrix(T_ba[:3, :3] @ T_true[:3, :3].T).magnitude())
    lin = np.linalg.norm(T_ba[:3, 3] - T_true[:3, 3])
    return ang, lin, spread, info


def test_exact():
    """Noise-free: the solve must return the rig it was given, to numerical precision."""
    rig_true = StereoRig.from_spherical(elev_deg=(45, 45), azim_deg=(0, 90), range_mm=300)
    ang, lin, _, _ = _recover(SPEC, rig_true, n=18, seed=1)
    print(f"  rotation error {ang:.6f} deg, translation error {lin:.6f} mm")
    assert ang < 0.01, f"rotation off by {ang} deg"
    assert lin < 0.01, f"translation off by {lin} mm"


def test_noise():
    """0.2 px corner noise: still sub-0.1 deg, and the spread reflects the noise."""
    rig_true = StereoRig.from_spherical(elev_deg=(45, 45), azim_deg=(0, 90), range_mm=300)
    base = rig_true.baseline_mm()
    ang, lin, spread, _ = _recover(SPEC, rig_true, n=24, seed=2, noise_px=0.2)
    print(f"  rotation error {ang:.4f} deg, translation error {lin:.4f} mm "
          f"({lin / base * 100:.3f}% of a {base:.0f} mm baseline)")
    assert ang < 0.5, f"rotation off by {ang} deg under 0.2 px noise"
    assert lin / base < 0.02, f"translation off by {lin / base * 100:.2f}% of baseline"
    assert spread["rot_deg_max"] > 0, "spread should be non-zero under noise"


def test_disjoint_ids():
    """Each camera sees a different corner subset -- the intersection must line them up."""
    rig_true = StereoRig.from_spherical(elev_deg=(45, -45), azim_deg=(0, 90), range_mm=300)
    ang, lin, _, _ = _recover(SPEC, rig_true, n=24, seed=3, drop=0.35)
    print(f"  rotation error {ang:.6f} deg, translation error {lin:.6f} mm")
    assert ang < 0.01, f"rotation off by {ang} deg with disjoint ids"
    assert lin < 0.01, f"translation off by {lin} mm with disjoint ids"


def test_units_are_mm():
    """A 9x6 at 16.667 mm spans 150 x 100 mm, not 0.15 x 0.10."""
    w, h = SPEC.size_mm
    print(f"  board {w:.3f} x {h:.3f} mm, {SPEC.n_corners} corners")
    assert 149 < w < 151 and 99 < h < 101, f"board is {w}x{h}, not millimetres"
    assert SPEC.n_corners == 40


def test_marker_scales_with_square():
    """A rescaled print rescales both dimensions."""
    s = SPEC.with_square_mm(SPEC.square_mm * 0.97)
    print(f"  {SPEC.marker_mm:.4f} -> {s.marker_mm:.4f} mm at 97% scale")
    assert abs(s.marker_mm / s.square_mm - SPEC.marker_mm / SPEC.square_mm) < 1e-12


for fn in (test_units_are_mm, test_marker_scales_with_square,
           test_exact, test_noise, test_disjoint_ids):
    print(f"{fn.__name__}: {fn.__doc__.splitlines()[0]}")
    fn()
    print("  ok\n")
print("all self-tests passed")

## 13. Intrinsics regression

`vision/board_images/9x6/` holds 29 single-camera shots at 960 × 720, one of which
(`image100.jpg`) detects zero corners. They cannot give an extrinsic — one camera — but they
do check this notebook's detection and board definition against the notebook that made the
checked-in `vision/camera_intrinsics.npz`.

Three numbers, measured here rather than asserted:

| path | fx | vs checked-in |
|---|---|---|
| this notebook — `IMREAD_GRAYSCALE`, as `sources.ImageSource` reads | 1411.143 | +2.36 |
| `visual_servo.ipynb` algorithm, same 28 images — `cvtColor(BGR2GRAY)` | 1410.801 | +2.02 |
| checked-in `camera_intrinsics.npz` | 1408.783 | — |

Two separate findings, worth keeping apart:

- **The 0.34 px between the first two rows is the JPEG grayscale decode path.**
  `cv2.imread(..., IMREAD_GRAYSCALE)` and `cvtColor(imread(...), BGR2GRAY)` disagree by up to
  **2 grey levels** (mean 0.005), which is enough to move the sub-pixel corners and shift fx
  by a third of a pixel. Board units (mm vs metres) and array shape (`(N,2)` vs `(N,1,2)`)
  were also checked and change K by *exactly* nothing. This notebook keeps
  `IMREAD_GRAYSCALE` to match how `sources.ImageSource` feeds the live pipeline.
- **The remaining ~2 px to the checked-in npz is not reproducible.** Neither decode path
  recovers it from the images now on disk, so that file predates this image set or was made
  with a different OpenCV. It is a fact about the reference, not a drift in this notebook —
  which is the whole reason this cell computes both paths instead of trusting one.

In [ ]:
def regression_9x6():
    ref_path = VISION / "camera_intrinsics.npz"
    img_dir = VISION / "board_images" / "9x6"
    if not ref_path.exists() or not img_dir.exists():
        print("reference intrinsics or board_images/9x6 missing -- skipping")
        return
    K_ref = np.asarray(np.load(ref_path)["camera_matrix"], dtype=np.float64)

    K_gray, _, _ = intrinsics_from_dir(img_dir, pattern="*.jpg", decode="gray",
                                       name="IMREAD_GRAYSCALE")
    print()
    K_bgr, _, _ = intrinsics_from_dir(img_dir, pattern="*.jpg", decode="bgr2gray",
                                      name="cvtColor BGR2GRAY")

    print(f"\n  {'':4s} {'IMREAD_GRAY':>13s} {'BGR2GRAY':>13s} {'checked in':>13s}"
          f" {'decode':>9s} {'vs ref':>9s}")
    for i, (label, r, c) in enumerate([("fx", 0, 0), ("fy", 1, 1), ("cx", 0, 2), ("cy", 1, 2)]):
        g, b, ref = K_gray[r, c], K_bgr[r, c], K_ref[r, c]
        print(f"  {label:4s} {g:13.3f} {b:13.3f} {ref:13.3f} {g - b:+9.3f} {g - ref:+9.3f}")

    print("\n  'decode' is IMREAD_GRAYSCALE minus BGR2GRAY on the SAME images: the two JPEG\n"
          "  grayscale paths disagree by up to 2 grey levels, which moves the sub-pixel\n"
          "  corners. Board units and array shape were checked separately and change K by\n"
          "  exactly zero, so this is the only difference between this notebook and the\n"
          "  algorithm in visual_servo.ipynb.\n\n"
          "  'vs ref' is larger and is NOT reproducible by either path, so the checked-in\n"
          "  npz predates these images or came from a different OpenCV. Treat it as a fact\n"
          "  about that file, not a drift here. If both decode columns agree with each\n"
          "  other, this notebook's detection and board definition are sound.")


regression_9x6()